# 🎓 French-Learning-Perceptions ML — Pipeline complet pas à pas
**Projet** : Représentations des élèves camerounais sur l'apprentissage du français  
**Objectif** : Suivre chaque étape ML de bout en bout — chargement → prétraitement → modélisation → évaluation  

---
### 📋 Étapes couvertes
1. ⚙️  Configuration & imports  
2. 📂 Chargement des données  
3. 🔍 Exploration des données (EDA)  
4. 🧹 Prétraitement  
5. 🏗️  Construction des features par hypothèse  
6. ✂️  Découpage Train / Validation / Test  
7. 🤖 Modélisation — H1, H2, H3, H4  
8. ✅ Évaluation & métriques  
9. 🔍 Interprétabilité (SHAP)  
10. 📊 MLflow — tracking des expériences  
11. 📝 Rapport final  


---
## ⚙️ Étape 1 — Configuration & imports
On installe les dépendances nécessaires et on configure l'environnement.

In [ ]:
# Vérifier que le venv est actif et les packages installés
import sys
print(f'Python : {sys.version}')
print(f'Chemin : {sys.executable}')

In [ ]:
# Imports principaux
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import mlflow
from pathlib import Path

# Affichage amélioré
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

print('✅ Imports OK')

In [ ]:
# Charger les paramètres depuis params.yaml
import sys
sys.path.insert(0, str(Path('..') / 'src'))

with open('../params.yaml', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

print('✅ params.yaml chargé')
print(f"  Données brutes  : {CFG['data']['raw_path']}")
print(f"  Random state    : {CFG['data']['random_state']}")
print(f"  Test size       : {CFG['data']['test_size']}")

---
## 📂 Étape 2 — Chargement des données
On charge le fichier CSV brut et on inspecte sa structure.

In [ ]:
# Charger le CSV brut
RAW_PATH = Path('..') / 'data' / 'raw' / 'data_FLP.csv'

df_raw = pd.read_csv(RAW_PATH)

print(f'✅ Données chargées')
print(f'   Lignes   : {df_raw.shape[0]}')
print(f'   Colonnes : {df_raw.shape[1]}')

In [ ]:
# Afficher les premières lignes
df_raw.head(3)

In [ ]:
# Liste de toutes les colonnes
for i, col in enumerate(df_raw.columns, 1):
    print(f'{i:2d}. {col[:80]}')

In [ ]:
# Vérifier le consentement
consent_col = [c for c in df_raw.columns if 'consentement' in c.lower() or 'accepte' in c.lower()][0]
print(f'Colonne consentement : {consent_col[:60]}')
print(df_raw[consent_col].value_counts())

---
## 🔍 Étape 3 — Exploration des données (EDA)
On explore la distribution des variables clés avant tout traitement.

In [ ]:
# Taux de valeurs manquantes par colonne
nan_rates = df_raw.isnull().mean().sort_values(ascending=False)
nan_rates = nan_rates[nan_rates > 0]

plt.figure(figsize=(10, 4))
nan_rates.plot(kind='bar', color='#2E75B6')
plt.title('Taux de valeurs manquantes par colonne', fontsize=13)
plt.ylabel('Proportion manquante')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

print(f'Colonnes avec NaN : {len(nan_rates)}/{df_raw.shape[1]}')

In [ ]:
# Distribution régionale
region_col = [c for c in df_raw.columns if 'gion' in c][0]
age_col    = [c for c in df_raw.columns if 'ge' in c.lower() and len(c) < 20][0]
sexe_col   = [c for c in df_raw.columns if 'sexe' in c.lower()][0]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Régions
df_raw[region_col].value_counts().plot(kind='bar', ax=axes[0], color='#2E75B6')
axes[0].set_title('Distribution régionale')
axes[0].tick_params(axis='x', rotation=30)

# Sexe
df_raw[sexe_col].value_counts().plot(kind='pie', ax=axes[1],
    autopct='%1.0f%%', colors=['#2E75B6','#F4A460'])
axes[1].set_title('Répartition par sexe')

# Âge (après nettoyage basique)
import re
ages = df_raw[age_col].apply(
    lambda x: float(re.search(r'\d+', str(x)).group()) if pd.notna(x) and re.search(r'\d+', str(x)) else np.nan
)
axes[2].hist(ages.dropna(), bins=10, color='#2E75B6', edgecolor='white')
axes[2].set_title('Distribution des âges')
axes[2].set_xlabel('Âge')

plt.suptitle('Profil démographique des répondants', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Perception du français (H2 — variable clé)
perc_col = [c for c in df_raw.columns if 'percevez' in c.lower()][0]

perc_counts = df_raw[perc_col].value_counts()

plt.figure(figsize=(8, 4))
perc_counts.plot(kind='barh', color='#2E75B6')
plt.title('Comment les élèves perçoivent-ils le français ? (H2)', fontsize=12)
plt.xlabel('Nombre de répondants')
plt.tight_layout()
plt.show()

In [ ]:
# Exposition aux autres langues (H3 — variable clé)
expo_col = [c for c in df_raw.columns if 'occasion' in c.lower()][0]

expo_counts = df_raw[expo_col].value_counts()

plt.figure(figsize=(8, 4))
expo_counts.plot(kind='bar', color='#1F4E79')
plt.title("Fréquence d'exposition aux autres langues (H3)", fontsize=12)
plt.ylabel('Nombre de répondants')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 🧹 Étape 4 — Prétraitement
Nettoyage, renommage des colonnes, anonymisation, normalisation des valeurs.

In [ ]:
from utils.constants import CSV_COLUMN_MAP, FREQ_MAP, LIKERT_MAP, IMPORTANCE_MAP
from preprocess import (
    load_and_rename, filter_consent, anonymize, clean_demographics
)

# 1. Renommer les colonnes
df = load_and_rename('../data/raw/data_FLP.csv')
print(f'1. Renommage : {df.shape[1]} colonnes mappées')

In [ ]:
# 2. Filtre consentement
n_avant = len(df)
df = filter_consent(df, CFG['data']['consent_value'])
print(f'2. Consentement : {len(df)}/{n_avant} répondants valides conservés')

In [ ]:
# 3. Anonymisation
cols_avant = set(df.columns)
df = anonymize(df)
supprimees = cols_avant - set(df.columns)
print(f'3. Anonymisation : colonnes supprimées → {supprimees}')

In [ ]:
# 4. Nettoyage démographique (âge, sexe, région)
df = clean_demographics(df)
print(f'4. Démographie nettoyée')
print(f'   Âge médian : {df["age"].median():.0f} ans')
print(f'   Sexe binaire (0=F, 1=M) : {df["sexe_bin"].value_counts().to_dict()}')

In [ ]:
# Aperçu après prétraitement
print('Colonnes disponibles après prétraitement :')
print([c for c in df.columns if not c.startswith('region_')])

---
## 🏗️ Étape 5 — Construction des features par hypothèse
Chaque hypothèse (H1→H4) génère son propre jeu de features à partir des indicateurs de la table French-Learning-Perceptions.

### H1 — Répertoire multilingue → Mobilisation
**VI** : nb_langues, langue_maternelle, apprentissage_antérieur, relation_LM  
**VD** : domaine_usage_freq, valorisation_sent  
**Cible** : usage_quotidien (OUI=1 / NON=0)

In [ ]:
from preprocess import build_h1

df_h1 = build_h1(df)

print(f'H1 — {len(df_h1)} lignes | {len(df_h1.columns)} colonnes')
print(f'\nDistribution cible (usage_quotidien) :')
print(df_h1['h1_target'].value_counts().rename({0: 'NON', 1: 'OUI'}))

# Features disponibles
h1_feat = [c for c in df_h1.columns if c not in ['h1_target'] and
           any(p in c for p in ['nb_langues','lm_','apprent','relation','domaine','valorisation',
                                'region_','age','sexe'])]
print(f'\nFeatures H1 ({len(h1_feat)}) :')
print(h1_feat)

### H2 — Représentations → Motivation & Difficultés
**VI** : perception_fr, mots_associés, importance_fr, hiérarchisation  
**Cible A** : motivation (0=Faible, 1=Moyen, 2=Élevé)  
**Cible B** : difficultés multi-label (grammaire, vocabulaire...)

In [ ]:
from preprocess import build_h2

df_h2 = build_h2(df)

print(f'H2 — {len(df_h2)} lignes')
print(f'\nCible A — Motivation :')
print(df_h2['h2_target_motivation'].value_counts().rename({0:'Faible',1:'Moyen',2:'Élevé'}))

print(f'\nCible B — Difficultés (multi-label) :')
diff_cols = [c for c in df_h2.columns if c.startswith('diff_')]
print(df_h2[diff_cols].sum().sort_values(ascending=False))

### H3 — Exposition plurilingue → Attitudes
**VI** : exposition_freq, intérêt autres langues, perception plurilinguisme  
**Cible** : score_attitude [1-5] + classe (Positive/Neutre/Négative)

In [ ]:
from preprocess import build_h3

df_h3 = build_h3(df)

print(f'H3 — {len(df_h3)} lignes')
print(f'\nScore attitude :')
print(df_h3['h3_score_attitude'].describe().round(3))

print(f'\nDistribution classes :')
print(df_h3['h3_attitude_class'].value_counts())

# Visualisation score vs exposition
plt.figure(figsize=(8,4))
for cls, color in [('Positive','#1F4E79'),('Neutre','#2E75B6'),('Négative','#F4A460')]:
    sub = df_h3[df_h3['h3_attitude_class'] == cls]
    plt.scatter(sub['exposition_freq'], sub['h3_score_attitude'],
                label=cls, alpha=0.6, color=color)
plt.xlabel('Fréquence exposition (Ind.1 VI)')
plt.ylabel('Score attitude envers le français')
plt.title('H3 : Exposition vs Attitude', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

### H4 — Intégration langues locales → Engagement
**VI** : intérêt camarades, souhait_freq, discipline souhaitée  
**Cible A** : motivation_accrue (OUI/NON)  
**Cible B** : engagement score [1-4]  
**Cible C** : discipline préférée (multi-label)

In [ ]:
from preprocess import build_h4

df_h4 = build_h4(df)

print(f'H4 — {len(df_h4)} lignes')
print(f'\nCible A — Motivation accrue :')
print(df_h4['h4_target_motivation'].value_counts().rename({0:'NON',1:'OUI'}))

print(f'\nCible B — Engagement score :')
print(df_h4['h4_engagement_score'].value_counts().sort_index())

print(f'\nCible C — Disciplines préférées :')
disc_cols = [c for c in df_h4.columns if c.startswith('vd_disc_')]
print(df_h4[disc_cols].sum().sort_values(ascending=False))

---
## ✂️ Étape 6 — Découpage Train / Validation / Test
Règle : **70% train | 15% validation | 15% test**  
⚠️ Stratification obligatoire pour préserver la distribution des classes.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM = CFG['data']['random_state']
TEST_S = CFG['data']['test_size']
VAL_S  = CFG['data']['val_size']

# ── H1 ──────────────────────────────────────────────────────────
h1_feat_cols = (
    [c for c in ['nb_langues','apprent_anterieur_bin','relation_lm_ord',
                  'domaine_usage_freq','valorisation_sent','sexe_bin','age']
     if c in df_h1.columns]
    + [c for c in df_h1.columns if c.startswith(('lm_','region_'))]
)
X_h1, y_h1 = df_h1[h1_feat_cols].fillna(0), df_h1['h1_target']

X_h1_tr, X_h1_te, y_h1_tr, y_h1_te = train_test_split(
    X_h1, y_h1, test_size=TEST_S, stratify=y_h1, random_state=RANDOM)
X_h1_tr, X_h1_val, y_h1_tr, y_h1_val = train_test_split(
    X_h1_tr, y_h1_tr, test_size=VAL_S/(1-TEST_S), stratify=y_h1_tr, random_state=RANDOM)

print('H1 — Splits :')
print(f'  Train      : {len(X_h1_tr):>4} lignes | OUI={y_h1_tr.mean():.1%}')
print(f'  Validation : {len(X_h1_val):>4} lignes | OUI={y_h1_val.mean():.1%}')
print(f'  Test       : {len(X_h1_te):>4} lignes | OUI={y_h1_te.mean():.1%}')

In [ ]:
# Vérification visuelle des splits
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, y) in zip(axes, [('Train', y_h1_tr),
                                  ('Validation', y_h1_val),
                                  ('Test', y_h1_te)]):
    y.value_counts().rename({0:'NON',1:'OUI'}).plot(
        kind='bar', ax=ax, color=['#F4A460','#1F4E79'], edgecolor='white')
    ax.set_title(f'{name} ({len(y)} lignes)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
plt.suptitle('H1 — Distribution OUI/NON dans chaque split', fontsize=12)
plt.tight_layout()
plt.show()

---
## 🤖 Étape 7 — Modélisation
On entraîne les modèles pour chaque hypothèse, du plus simple au plus complexe.

### 7.1 — H1 : Classification binaire (mobilisation des langues)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

# SMOTE sur train uniquement (corriger le déséquilibre OUI/NON)
sm = SMOTE(random_state=RANDOM)
X_h1_tr_res, y_h1_tr_res = sm.fit_resample(X_h1_tr, y_h1_tr)
print(f'Après SMOTE : {len(X_h1_tr_res)} lignes train (équilibré)')

In [ ]:
# Baseline — Régression Logistique
baseline_h1 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LogisticRegression(max_iter=500, random_state=RANDOM))
])
baseline_h1.fit(X_h1_tr_res, y_h1_tr_res)
print('✅ Baseline H1 (LogReg) entraîné')

In [ ]:
# Modèle principal — XGBoost
c_h1 = CFG['h1']

model_h1 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   xgb.XGBClassifier(
        n_estimators   = c_h1['n_estimators'],
        max_depth      = c_h1['max_depth'],
        learning_rate  = c_h1['learning_rate'],
        subsample      = c_h1['subsample'],
        colsample_bytree = c_h1['colsample_bytree'],
        use_label_encoder=False,
        eval_metric    = 'logloss',
        random_state   = RANDOM
    ))
])

model_h1.fit(X_h1_tr_res, y_h1_tr_res)
print('✅ XGBoost H1 entraîné')

### 7.2 — H2 : Multi-output (motivation + difficultés)

In [ ]:
from sklearn.multioutput import MultiOutputClassifier

h2_feat_cols = (
    [c for c in ['perc_utile','perc_belle','perc_difficile','perc_importante',
                  'mots_assoc_sent','importance_bin','importance_sent',
                  'hierarchie_fr','age','sexe_bin'] if c in df_h2.columns]
    + [c for c in df_h2.columns if c.startswith('region_')]
)
diff_cols = [c for c in df_h2.columns if c.startswith('diff_')]

X_h2 = df_h2[h2_feat_cols].fillna(0)
y_h2A = df_h2['h2_target_motivation']
y_h2B = df_h2[diff_cols].fillna(0).astype(int)

X_h2_tr, X_h2_te, yA_tr, yA_te, yB_tr, yB_te = train_test_split(
    X_h2, y_h2A, y_h2B, test_size=TEST_S, stratify=y_h2A, random_state=RANDOM)

# Cible A — motivation (3 classes)
model_h2A = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   xgb.XGBClassifier(
        n_estimators=CFG['h2']['n_estimators'],
        max_depth=CFG['h2']['max_depth'],
        use_label_encoder=False, eval_metric='mlogloss',
        num_class=3, objective='multi:softprob', random_state=RANDOM))
])
model_h2A.fit(X_h2_tr, yA_tr)

# Cible B — difficultés (multi-label)
model_h2B = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   MultiOutputClassifier(
        xgb.XGBClassifier(n_estimators=CFG['h2']['n_estimators'],
                           use_label_encoder=False, eval_metric='logloss',
                           random_state=RANDOM)))
])
model_h2B.fit(X_h2_tr, yB_tr)

print('✅ H2 — Cible A (motivation) + Cible B (difficultés) entraînées')

### 7.3 — H3 : Régression + Classification + Analyse causale

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from scipy.stats import pearsonr

h3_feat_cols = (
    [c for c in ['exposition_freq','interet_bin','interet_sent',
                  'perception_multi_sent','perception_multi_ord',
                  'nb_langues','age','sexe_bin'] if c in df_h3.columns]
    + [c for c in df_h3.columns if c.startswith('region_')]
)

le_h3 = LabelEncoder().fit(['Négative','Neutre','Positive'])
X_h3 = df_h3[h3_feat_cols].fillna(0)
y_h3_reg = df_h3['h3_score_attitude']
y_h3_clf = le_h3.transform(df_h3['h3_attitude_class'].fillna('Neutre'))

X_h3_tr, X_h3_te, yr_tr, yr_te, yc_tr, yc_te = train_test_split(
    X_h3, y_h3_reg, y_h3_clf, test_size=TEST_S, random_state=RANDOM)

# Régression
model_h3_reg = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   RandomForestRegressor(
        n_estimators=CFG['h3']['n_estimators_reg'],
        max_depth=CFG['h3']['max_depth'], random_state=RANDOM))
])
model_h3_reg.fit(X_h3_tr, yr_tr)

# Classification
model_h3_clf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   xgb.XGBClassifier(
        n_estimators=CFG['h3']['n_estimators_clf'],
        use_label_encoder=False, eval_metric='mlogloss',
        num_class=3, objective='multi:softprob', random_state=RANDOM))
])
model_h3_clf.fit(X_h3_tr, yc_tr)

print('✅ H3 — Régression + Classification entraînées')

### 7.4 — H4 : Multi-label + Ordinal + Préférences

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from scipy.stats import spearmanr

h4_feat_cols = (
    [c for c in ['interet_camarades_bin','interet_camarades_sent',
                  'souhait_freq','age','sexe_bin'] if c in df_h4.columns]
    + [c for c in df_h4.columns if c.startswith(('vi_disc_','region_'))]
)
disc_cols = [c for c in df_h4.columns if c.startswith('vd_disc_')]

X_h4 = df_h4[h4_feat_cols].fillna(0)
y_h4A = df_h4['h4_target_motivation'].fillna(0).astype(int)
y_h4B = df_h4['h4_engagement_score'].fillna(1).astype(int)
y_h4C = df_h4[disc_cols].fillna(0).astype(int)

X_h4_tr, X_h4_te, yA_tr, yA_te, yB_tr, yB_te, yC_tr, yC_te = train_test_split(
    X_h4, y_h4A, y_h4B, y_h4C, test_size=TEST_S, stratify=y_h4A, random_state=RANDOM)

# Cible A — motivation accrue (binaire)
model_h4A = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   VotingClassifier([
        ('xgb', xgb.XGBClassifier(use_label_encoder=False,
                                    eval_metric='logloss', random_state=RANDOM)),
        ('lr',  LogisticRegression(max_iter=500, class_weight='balanced'))
    ], voting='soft'))
])
model_h4A.fit(X_h4_tr, yA_tr)

# Cible B — engagement ordinal
model_h4B = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   xgb.XGBClassifier(
        n_estimators=CFG['h4']['n_estimators'],
        use_label_encoder=False, eval_metric='mlogloss',
        num_class=4, objective='multi:softprob', random_state=RANDOM))
])
model_h4B.fit(X_h4_tr, y_h4B.iloc[X_h4_tr.index] - 1 if hasattr(X_h4_tr,'index') else yB_tr - 1)

# Cible C — discipline préférée (multi-label)
model_h4C = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   MultiOutputClassifier(
        xgb.XGBClassifier(use_label_encoder=False,
                           eval_metric='logloss', random_state=RANDOM)))
])
model_h4C.fit(X_h4_tr, yC_tr)

print('✅ H4 — Cibles A + B + C entraînées')

---
## ✅ Étape 8 — Évaluation & Métriques
On calcule les métriques sur le **test set** et on vérifie si les seuils sont atteints.

In [ ]:
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    mean_absolute_error, accuracy_score, confusion_matrix
)

# ── H1 ──────────────────────────────────────────────────────────
y_h1_pred  = model_h1.predict(X_h1_te)
y_h1_proba = model_h1.predict_proba(X_h1_te)[:, 1]
f1_h1  = f1_score(y_h1_te, y_h1_pred, average='macro')
auc_h1 = roc_auc_score(y_h1_te, y_h1_proba)

print('═'*50)
print('H1 — Mobilisation des langues')
print('═'*50)
print(f'  F1-macro : {f1_h1:.3f}  (seuil ≥ {CFG["h1"]["thresholds"]["f1_macro"]}) '
      f'{"✅" if f1_h1 >= CFG["h1"]["thresholds"]["f1_macro"] else "⚠️"}')
print(f'  ROC-AUC  : {auc_h1:.3f}  (seuil ≥ {CFG["h1"]["thresholds"]["roc_auc"]}) '
      f'{"✅" if auc_h1 >= CFG["h1"]["thresholds"]["roc_auc"] else "⚠️"}')
print()
print(classification_report(y_h1_te, y_h1_pred, target_names=['NON','OUI']))

In [ ]:
# Matrice de confusion H1
cm = confusion_matrix(y_h1_te, y_h1_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NON','OUI'], yticklabels=['NON','OUI'])
plt.title('H1 — Matrice de confusion', fontsize=12)
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.show()

In [ ]:
# ── H2 ──────────────────────────────────────────────────────────
yA_pred = model_h2A.predict(X_h2_te)
yB_pred = model_h2B.predict(X_h2_te)
f1_h2A  = f1_score(yA_te, yA_pred, average='weighted')
f1_h2B  = f1_score(yB_te, yB_pred, average='micro', zero_division=0)

print('═'*50)
print('H2 — Représentations → Motivation & Difficultés')
print('═'*50)
print(f'  Cible A F1-weighted : {f1_h2A:.3f}  '
      f'(seuil ≥ {CFG["h2"]["thresholds"]["f1_weighted_A"]}) '
      f'{"✅" if f1_h2A >= CFG["h2"]["thresholds"]["f1_weighted_A"] else "⚠️"}')
print(f'  Cible B F1-micro    : {f1_h2B:.3f}  '
      f'(seuil ≥ {CFG["h2"]["thresholds"]["f1_micro_B"]}) '
      f'{"✅" if f1_h2B >= CFG["h2"]["thresholds"]["f1_micro_B"] else "⚠️"}')

In [ ]:
# ── H3 ──────────────────────────────────────────────────────────
yr_pred = model_h3_reg.predict(X_h3_te)
yc_pred = model_h3_clf.predict(X_h3_te)
mae_h3  = mean_absolute_error(yr_te, yr_pred)
f1_h3   = f1_score(yc_te, yc_pred, average='weighted')
r, p    = pearsonr(df_h3['exposition_freq'].fillna(0),
                    df_h3['h3_score_attitude'].fillna(3))

print('═'*50)
print('H3 — Exposition → Attitudes')
print('═'*50)
print(f'  MAE régression  : {mae_h3:.3f}  '
      f'(seuil ≤ {CFG["h3"]["thresholds"]["mae"]}) '
      f'{"✅" if mae_h3 <= CFG["h3"]["thresholds"]["mae"] else "⚠️"}')
print(f'  F1-weighted clf : {f1_h3:.3f}  '
      f'(seuil ≥ {CFG["h3"]["thresholds"]["f1_weighted"]}) '
      f'{"✅" if f1_h3 >= CFG["h3"]["thresholds"]["f1_weighted"] else "⚠️"}')
print(f'  Pearson r       : {r:.3f}  p={p:.4f}  '
      f'{"✅ Significatif" if p < 0.05 and r > 0 else "⚠️ Non significatif"}')

In [ ]:
# ── H4 ──────────────────────────────────────────────────────────
yA_h4_pred = model_h4A.predict(X_h4_te)
yB_h4_pred = model_h4B.predict(X_h4_te) + 1
yC_h4_pred = model_h4C.predict(X_h4_te)
f1_h4A     = f1_score(yA_te, yA_h4_pred)
rho, _     = spearmanr(yB_te, yB_h4_pred)
sub_acc    = accuracy_score(yC_te, yC_h4_pred)

print('═'*50)
print('H4 — Intégration langues locales → Engagement')
print('═'*50)
print(f'  Cible A F1        : {f1_h4A:.3f}  '
      f'(seuil ≥ {CFG["h4"]["thresholds"]["f1_A"]}) '
      f'{"✅" if f1_h4A >= CFG["h4"]["thresholds"]["f1_A"] else "⚠️"}')
print(f'  Cible B Spearman ρ: {rho:.3f}  '
      f'(seuil ≥ {CFG["h4"]["thresholds"]["spearman_B"]}) '
      f'{"✅" if rho >= CFG["h4"]["thresholds"]["spearman_B"] else "⚠️"}')
print(f'  Cible C Subset acc: {sub_acc:.3f}  '
      f'(seuil ≥ {CFG["h4"]["thresholds"]["subset_C"]}) '
      f'{"✅" if sub_acc >= CFG["h4"]["thresholds"]["subset_C"] else "⚠️"}')

---
## 🔍 Étape 9 — Interprétabilité (SHAP)
SHAP nous explique **quelles features influencent le plus les prédictions** de chaque modèle.

In [ ]:
import shap

# SHAP pour H1 (XGBoost)
explainer_h1 = shap.TreeExplainer(model_h1.named_steps['model'])
X_h1_scaled  = model_h1[:-1].transform(X_h1_te)
shap_vals    = explainer_h1.shap_values(X_h1_scaled)

print('H1 — Top 10 features par importance SHAP :')
mean_shap = np.abs(shap_vals).mean(axis=0)
top10 = sorted(zip(h1_feat_cols, mean_shap), key=lambda x: x[1], reverse=True)[:10]
for i, (feat, val) in enumerate(top10, 1):
    bar = '█' * int(val * 30 / top10[0][1])
    print(f'  {i:2d}. {feat:<35} {bar} {val:.4f}')

In [ ]:
# Visualisation SHAP — beeswarm plot H1
shap.summary_plot(
    shap_vals,
    X_h1_scaled,
    feature_names=h1_feat_cols,
    max_display=10,
    plot_type='bar',
    show=True
)

---
## 📊 Étape 10 — Tracking MLflow
On log tous les résultats dans MLflow pour comparer les runs et garder une trace.

In [ ]:
import mlflow
from datetime import datetime

mlflow.set_tracking_uri('../mlruns')
ts = datetime.now().strftime('%Y%m%d_%H%M')

# ── Log H1 ──────────────────────────────────────────────────────
mlflow.set_experiment('flp_H1_mobilisation')
with mlflow.start_run(run_name=f'notebook_H1_{ts}'):
    # Paramètres
    mlflow.log_params({
        'n_estimators': CFG['h1']['n_estimators'],
        'max_depth':    CFG['h1']['max_depth'],
        'smote':        CFG['h1']['smote'],
        'train_size':   len(X_h1_tr)
    })
    # Métriques
    mlflow.log_metric('f1_macro', f1_h1)
    mlflow.log_metric('roc_auc',  auc_h1)
    mlflow.log_metric('threshold_f1_macro', int(f1_h1  >= CFG['h1']['thresholds']['f1_macro']))
    mlflow.log_metric('threshold_roc_auc',  int(auc_h1 >= CFG['h1']['thresholds']['roc_auc']))
    # Tag
    status = '✅ OK' if (f1_h1 >= CFG['h1']['thresholds']['f1_macro'] and
                         auc_h1 >= CFG['h1']['thresholds']['roc_auc']) else '⚠️ KO'
    mlflow.set_tag('status', status)
    mlflow.set_tag('source', 'notebook')

print(f'✅ H1 loggé dans MLflow — statut : {status}')
print('\nPour voir les résultats :')
print('  Terminal → mlflow ui --backend-store-uri ../mlruns --port 5000')
print('  Navigateur → http://localhost:5000')

---
## 📝 Étape 11 — Rapport final
Synthèse des résultats pour toutes les hypothèses.

In [ ]:
SEUILS = {
    'H1': {'F1-macro': (f1_h1,  CFG['h1']['thresholds']['f1_macro'],  '>='),
           'ROC-AUC':  (auc_h1, CFG['h1']['thresholds']['roc_auc'],   '>=')},
    'H2': {'F1-A':     (f1_h2A, CFG['h2']['thresholds']['f1_weighted_A'], '>='),
           'F1-B':     (f1_h2B, CFG['h2']['thresholds']['f1_micro_B'],    '>=')},
    'H3': {'MAE':      (mae_h3, CFG['h3']['thresholds']['mae'],         '<='),
           'F1-clf':   (f1_h3,  CFG['h3']['thresholds']['f1_weighted'],  '>='),
           'Pearson-r':(r,      CFG['h3']['thresholds']['pearson_p'],    'p<')},
    'H4': {'F1-A':     (f1_h4A, CFG['h4']['thresholds']['f1_A'],        '>='),
           'Spearman': (rho,    CFG['h4']['thresholds']['spearman_B'],   '>='),
           'Subset':   (sub_acc,CFG['h4']['thresholds']['subset_C'],     '>=')},
}

print('\n' + '═'*60)
print('       RAPPORT FINAL — FRENCH-LEARNING-PERCEPTIONS ML')
print('═'*60)

n_validated = 0
for hyp, metrics in SEUILS.items():
    all_ok = True
    print(f'\n  {hyp}')
    for name, (val, seuil, op) in metrics.items():
        if op == '<=':
            ok = val <= seuil
        elif op == 'p<':
            ok = p < seuil and r > 0
        else:
            ok = val >= seuil
        icon = '✅' if ok else '⚠️'
        if not ok: all_ok = False
        print(f'    {icon} {name:<15} {val:.3f}  (seuil {op} {seuil})')
    if all_ok: n_validated += 1

print(f'\n═'*60)
print(f'  Hypothèses validées : {n_validated}/4')
status = '✅ Projet opérationnel' if n_validated >= 3 else '⚠️ Affiner les modèles'
print(f'  Statut : {status}')
print('═'*60)

In [ ]:
# Graphique récapitulatif des métriques clés
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('French-Learning-Perceptions ML — Métriques par hypothèse', fontsize=14)

configs = [
    ('H1', axes[0,0], ['F1-macro','ROC-AUC'],
     [f1_h1, auc_h1],
     [CFG['h1']['thresholds']['f1_macro'], CFG['h1']['thresholds']['roc_auc']]),
    ('H2', axes[0,1], ['F1-A (motiv.)','F1-B (diff.)'],
     [f1_h2A, f1_h2B],
     [CFG['h2']['thresholds']['f1_weighted_A'], CFG['h2']['thresholds']['f1_micro_B']]),
    ('H3', axes[1,0], ['MAE (inv.)','F1-clf'],
     [1-mae_h3, f1_h3],
     [1-CFG['h3']['thresholds']['mae'], CFG['h3']['thresholds']['f1_weighted']]),
    ('H4', axes[1,1], ['F1-A','Spearman ρ','Subset acc'],
     [f1_h4A, rho, sub_acc],
     [CFG['h4']['thresholds']['f1_A'], CFG['h4']['thresholds']['spearman_B'],
      CFG['h4']['thresholds']['subset_C']]),
]

for hyp, ax, labels, vals, seuils in configs:
    colors = ['#1F4E79' if v >= s else '#F4A460' for v, s in zip(vals, seuils)]
    bars = ax.bar(labels, vals, color=colors, edgecolor='white')
    for s, x in zip(seuils, range(len(seuils))):
        ax.axhline(s, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(f'{hyp}', fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
print('Bleu = seuil atteint ✅ | Orange = seuil non atteint ⚠️ | Ligne rouge = seuil')

---
## 🚀 Prochaines étapes
```
✅ Level 0  — Scripts manuels + tests
✅ Level 1  — Pipeline automatisé + MLflow tracking
🔲 Level 2  — GitHub Actions CI/CD + FastAPI /predict
🔲 Level 3  — Data drift + retraining automatique
```

Pour lancer le pipeline complet depuis le terminal VS Code :
```powershell
python src/pipeline.py --config params.yaml
mlflow ui --port 5000
```
